In [1]:
# Imports
import os
import polars as pl
import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re

# # Opciones de visualización
# %matplotlib inline
# plt.rcParams['figure.figsize'] = (10,5)

In [2]:
path = '../results/championship/'
files = os.listdir(path)
regex = re.compile(r'(\d+_\d+)_(.*).parquet')

In [3]:
dict_edition_phase = {}
for file in files:
    match = regex.match(file)
    if match:
        edition = match.group(1)
        phase = match.group(2)
        df = pl.read_parquet(f'{path}{file}')
        if edition not in dict_edition_phase:
            dict_edition_phase[edition] = {phase: df}
        else:
            dict_edition_phase[edition][phase] = df

        print(f'Edition: {edition}, Phase: {phase}')

Edition: 20251207_110916, Phase: first_phase
Edition: 20251207_110916, Phase: second_phase
Edition: 20251207_110916, Phase: third_phase
Edition: 20251208_114259, Phase: first_phase
Edition: 20251208_114259, Phase: second_phase
Edition: 20251208_114259, Phase: third_phase
Edition: 20251208_114346, Phase: first_phase
Edition: 20251208_114346, Phase: second_phase
Edition: 20251208_114346, Phase: third_phase
Edition: 20251208_114408, Phase: first_phase
Edition: 20251208_114408, Phase: second_phase
Edition: 20251208_114408, Phase: third_phase
Edition: 20251208_114426, Phase: first_phase
Edition: 20251208_114426, Phase: second_phase
Edition: 20251208_114426, Phase: third_phase
Edition: 20251208_114444, Phase: first_phase
Edition: 20251208_114444, Phase: second_phase
Edition: 20251208_114444, Phase: third_phase
Edition: 20251208_114539, Phase: first_phase
Edition: 20251208_114539, Phase: second_phase
Edition: 20251208_114539, Phase: third_phase


In [4]:
edition = "20251207_110916"
sample_first = dict_edition_phase.get(edition).get("first_phase")
sample_second = dict_edition_phase.get(edition).get("second_phase")
sample_third = dict_edition_phase.get(edition).get("third_phase")

# Samples

In [19]:
sample_first

game_number,repetition,num_rounds,p1_name,p2_name,p1_action,p2_action,p1_payoff,p2_payoff,mean_score_p1,mean_score_p2
i64,i64,i64,str,str,list[i64],list[i64],list[i64],list[i64],f64,f64
1,1,1,"""Adaptive Pavlov""","""Agente Astuto""",[2],[2],[2],[2],2.0,2.0
2,2,35,"""Adaptive Pavlov""","""Agente Astuto""","[2, 2, … 2]","[2, 2, … 2]","[2, 2, … 2]","[2, 2, … 2]",1.971429,2.142857
3,1,100,"""Adaptive Pavlov""","""BinarySunset""","[2, 3, … 3]","[4, 4, … 2]","[0, 0, … 3]","[0, 0, … 2]",1.62,1.29
4,2,10,"""Adaptive Pavlov""","""BinarySunset""","[2, 3, … 3]","[4, 4, … 4]","[0, 0, … 0]","[0, 0, … 0]",0.0,0.0
5,1,49,"""Adaptive Pavlov""","""Contrite TFT""","[2, 2, … 2]","[2, 2, … 2]","[2, 2, … 2]","[2, 2, … 2]",2.0,2.020408
…,…,…,…,…,…,…,…,…,…,…
178,2,49,"""Random 2 or 3""","""Weighted Random 2, 3, or 4""","[2, 3, … 3]","[3, 2, … 2]","[2, 3, … 3]","[3, 2, … 2]",1.857143,1.755102
179,1,22,"""Random 2 or 3""","""WSLS""","[3, 3, … 2]","[2, 3, … 3]","[3, 0, … 2]","[2, 0, … 3]",1.5,1.5
180,2,48,"""Random 2 or 3""","""WSLS""","[3, 3, … 3]","[2, 3, … 3]","[3, 0, … 0]","[2, 0, … 0]",1.5,1.75


In [22]:
duckdb.query("""
SELECT game_number, p1_name, p2_name, mean_score_p1, mean_score_p2, list_avg(p1_payoff)
FROM sample_first
""")

┌─────────────┬────────────────────────────┬────────────────────────────┬────────────────────┬────────────────────┬─────────────────────┐
│ game_number │          p1_name           │          p2_name           │   mean_score_p1    │   mean_score_p2    │ list_avg(p1_payoff) │
│    int64    │          varchar           │          varchar           │       double       │       double       │       double        │
├─────────────┼────────────────────────────┼────────────────────────────┼────────────────────┼────────────────────┼─────────────────────┤
│           1 │ Adaptive Pavlov            │ Agente Astuto              │                2.0 │                2.0 │                 2.0 │
│           2 │ Adaptive Pavlov            │ Agente Astuto              │ 1.9714285714285715 │  2.142857142857143 │  1.9714285714285715 │
│           3 │ Adaptive Pavlov            │ BinarySunset               │               1.62 │               1.29 │                1.62 │
│           4 │ Adaptive Pavlov   

## Mayor media

Toneo y global



In [30]:
duckdb.query("""
SELECT p1_name as player_name, list_sum(p1_payoff) as player_score, num_rounds as players_rounds, player_score/players_rounds as mean , mean_score_p1    
FROM sample_first
""")

┌────────────────────────────┬──────────────┬────────────────┬────────────────────┬────────────────────┐
│        player_name         │ player_score │ players_rounds │        mean        │   mean_score_p1    │
│          varchar           │    int128    │     int64      │       double       │       double       │
├────────────────────────────┼──────────────┼────────────────┼────────────────────┼────────────────────┤
│ Adaptive Pavlov            │            2 │              1 │                2.0 │                2.0 │
│ Adaptive Pavlov            │           69 │             35 │ 1.9714285714285715 │ 1.9714285714285715 │
│ Adaptive Pavlov            │          162 │            100 │               1.62 │               1.62 │
│ Adaptive Pavlov            │            0 │             10 │                0.0 │                0.0 │
│ Adaptive Pavlov            │           98 │             49 │                2.0 │                2.0 │
│ Adaptive Pavlov            │          108 │          

In [67]:
# Primer torneo
duckdb.query("""
SELECT player_name, SUM(player_score) as total_score, SUM(num_rounds) as total_rounds, total_score/total_rounds as Average
FROM (
    SELECT p1_name as player_name, list_sum(p1_payoff) as player_score, num_rounds
    FROM sample_first
    UNION ALL
    SELECT p2_name as player_name, list_sum(p2_payoff) as player_score, num_rounds
    FROM sample_first
    )
GROUP BY player_name
ORDER BY Average DESC
""")

┌────────────────────────────┬─────────────┬──────────────┬────────────────────┐
│        player_name         │ total_score │ total_rounds │      Average       │
│          varchar           │   int128    │    int128    │       double       │
├────────────────────────────┼─────────────┼──────────────┼────────────────────┤
│ Adaptive Pavlov            │        4159 │         2124 │ 1.9580979284369116 │
│ Hat Tricker                │        5123 │         2706 │ 1.8932002956393201 │
│ Weighted Random 2, 3, or 4 │        2844 │         1528 │  1.861256544502618 │
│ Contrite TFT               │        4981 │         2766 │ 1.8007953723788865 │
│ Permissive Tit for Tat     │        4935 │         2950 │ 1.6728813559322033 │
│ WSLS                       │        4406 │         2681 │ 1.6434166355837374 │
│ Random 2 or 3              │        4901 │         3007 │ 1.6298636514798803 │
│ Agente Astuto              │        4213 │         2596 │ 1.6228813559322033 │
│ Generous TFT              

In [68]:
# Total
duckdb.query("""
SELECT player_name, SUM(player_score) as total_score, SUM(num_rounds) as total_rounds, total_score/total_rounds as Average
FROM (
    SELECT p1_name as player_name, list_sum(p1_payoff) as player_score, num_rounds
    FROM sample_first
    UNION ALL
    SELECT p2_name as player_name, list_sum(p2_payoff) as player_score, num_rounds
    FROM sample_first
    UNION ALL
    SELECT p1_name as player_name, list_sum(p1_payoff) as player_score, num_rounds
    FROM sample_second
    UNION ALL
    SELECT p2_name as player_name, list_sum(p2_payoff) as player_score, num_rounds
    FROM sample_second
    UNION ALL
    SELECT p1_name as player_name, list_sum(p1_payoff) as player_score, num_rounds
    FROM sample_third
    UNION ALL
    SELECT p2_name as player_name, list_sum(p2_payoff) as player_score, num_rounds
    FROM sample_third
    )
GROUP BY player_name
ORDER BY Average DESC
""")

┌────────────────────────────┬─────────────┬──────────────┬──────────────────────┐
│        player_name         │ total_score │ total_rounds │       Average        │
│          varchar           │   int128    │    int128    │        double        │
├────────────────────────────┼─────────────┼──────────────┼──────────────────────┤
│ Adaptive Pavlov            │      123822 │        63112 │   1.9619406768918748 │
│ Focal 5                    │       67812 │        35228 │    1.924946065629613 │
│ Castigador Infernal        │       67183 │        35823 │   1.8754152360215504 │
│ Hat Tricker                │      112528 │        61831 │   1.8199285148226618 │
│ Weighted Random 2, 3, or 4 │      112692 │        62422 │   1.8053250456569798 │
│ Contrite TFT               │      105426 │        62589 │   1.6844173896371566 │
│ Random 2 or 3              │      111819 │        67561 │   1.6550820739775907 │
│ WSLS                       │      104656 │        63246 │   1.6547449641084022 │
│ Ge

## Nemesis

In [44]:
duckdb.query("""
    SELECT 
        p1_name,
        p2_name,
        list_sum(p1_payoff) AS p1_points,
        list_sum(p2_payoff) AS p2_points,
        CASE 
            WHEN list_sum(p1_payoff) > list_sum(p2_payoff) THEN p1_name
            WHEN list_sum(p2_payoff) > list_sum(p1_payoff) THEN p2_name
            ELSE NULL
        END AS winner,
        CASE 
            WHEN list_sum(p1_payoff) < list_sum(p2_payoff) THEN p1_name
            WHEN list_sum(p2_payoff) < list_sum(p1_payoff) THEN p2_name
            ELSE NULL
        END AS Looser,
        list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
    FROM sample_first
""")

┌────────────────────────────┬────────────────────────────┬───────────┬───────────┬────────────────────────────┬────────────────────────────┬─────────┐
│          p1_name           │          p2_name           │ p1_points │ p2_points │           winner           │           Looser           │ is_draw │
│          varchar           │          varchar           │  int128   │  int128   │          varchar           │          varchar           │ boolean │
├────────────────────────────┼────────────────────────────┼───────────┼───────────┼────────────────────────────┼────────────────────────────┼─────────┤
│ Adaptive Pavlov            │ Agente Astuto              │         2 │         2 │ NULL                       │ NULL                       │ true    │
│ Adaptive Pavlov            │ Agente Astuto              │        69 │        75 │ Agente Astuto              │ Adaptive Pavlov            │ false   │
│ Adaptive Pavlov            │ BinarySunset               │       162 │       129 │ Adap

In [49]:
player = "Adaptive Pavlov"

In [52]:
duckdb.query(f"""
    SELECT Looser, Winner, count(*) as perdidos
    FROM (
        SELECT 
            p1_name,
            p2_name,
            list_sum(p1_payoff) AS p1_points,
            list_sum(p2_payoff) AS p2_points,
            CASE 
                WHEN list_sum(p1_payoff) > list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) > list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS winner,
            CASE 
                WHEN list_sum(p1_payoff) < list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) < list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS Looser,
            list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
        FROM sample_first
    )
    WHERE Looser = '{player}'
    GROUP BY Looser, Winner
    ORDER BY perdidos DESC
    """)

┌─────────────────┬────────────────────────────┬──────────┐
│     Looser      │           winner           │ perdidos │
│     varchar     │          varchar           │  int64   │
├─────────────────┼────────────────────────────┼──────────┤
│ Adaptive Pavlov │ WSLS                       │        2 │
│ Adaptive Pavlov │ Detective Avanzado         │        2 │
│ Adaptive Pavlov │ Grim Trigger               │        2 │
│ Adaptive Pavlov │ Random 2 or 3              │        2 │
│ Adaptive Pavlov │ Hat Tricker                │        2 │
│ Adaptive Pavlov │ Generous TFT               │        1 │
│ Adaptive Pavlov │ Weighted Random 2, 3, or 4 │        1 │
│ Adaptive Pavlov │ Contrite TFT               │        1 │
│ Adaptive Pavlov │ CopyCat                    │        1 │
│ Adaptive Pavlov │ Agente Astuto              │        1 │
│ Adaptive Pavlov │ Permissive Tit for Tat     │        1 │
├─────────────────┴────────────────────────────┴──────────┤
│ 11 rows                               

In [53]:
duckdb.query(f"""
    SELECT Looser, Winner, count(*) as perdidos
    FROM (
        SELECT 
            p1_name,
            p2_name,
            list_sum(p1_payoff) AS p1_points,
            list_sum(p2_payoff) AS p2_points,
            CASE 
                WHEN list_sum(p1_payoff) > list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) > list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS winner,
            CASE 
                WHEN list_sum(p1_payoff) < list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) < list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS Looser,
            list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
        FROM sample_first
        UNION ALL
        SELECT 
            p1_name,
            p2_name,
            list_sum(p1_payoff) AS p1_points,
            list_sum(p2_payoff) AS p2_points,
            CASE 
                WHEN list_sum(p1_payoff) > list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) > list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS winner,
            CASE 
                WHEN list_sum(p1_payoff) < list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) < list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS Looser,
            list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
        FROM sample_second
        UNION ALL
        SELECT 
            p1_name,
            p2_name,
            list_sum(p1_payoff) AS p1_points,
            list_sum(p2_payoff) AS p2_points,
            CASE 
                WHEN list_sum(p1_payoff) > list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) > list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS winner,
            CASE 
                WHEN list_sum(p1_payoff) < list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) < list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS Looser,
            list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
        FROM sample_third
    )
    WHERE Looser = '{player}'
    GROUP BY Looser, Winner
    ORDER BY perdidos DESC
    """)

┌─────────────────┬────────────────────────────┬──────────┐
│     Looser      │           winner           │ perdidos │
│     varchar     │          varchar           │  int64   │
├─────────────────┼────────────────────────────┼──────────┤
│ Adaptive Pavlov │ Detective Avanzado         │       42 │
│ Adaptive Pavlov │ Grim Trigger               │       42 │
│ Adaptive Pavlov │ Random 2 or 3              │       42 │
│ Adaptive Pavlov │ WSLS                       │       42 │
│ Adaptive Pavlov │ Hat Tricker                │       42 │
│ Adaptive Pavlov │ Weighted Random 2, 3, or 4 │       41 │
│ Adaptive Pavlov │ Agente Astuto              │       40 │
│ Adaptive Pavlov │ Deterministic Simpletron   │       33 │
│ Adaptive Pavlov │ Permissive Tit for Tat     │       24 │
│ Adaptive Pavlov │ CopyCat                    │       21 │
│ Adaptive Pavlov │ Focal 5                    │       20 │
│ Adaptive Pavlov │ Always 3                   │       20 │
│ Adaptive Pavlov │ Castigador Infernal 

In [54]:
duckdb.query(f"""
    SELECT Looser, Winner, count(*) as perdidos
    FROM (
        SELECT 
            p1_name,
            p2_name,
            list_sum(p1_payoff) AS p1_points,
            list_sum(p2_payoff) AS p2_points,
            CASE 
                WHEN list_sum(p1_payoff) > list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) > list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS winner,
            CASE 
                WHEN list_sum(p1_payoff) < list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) < list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS Looser,
            list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
        FROM sample_first
        UNION ALL
        SELECT 
            p1_name,
            p2_name,
            list_sum(p1_payoff) AS p1_points,
            list_sum(p2_payoff) AS p2_points,
            CASE 
                WHEN list_sum(p1_payoff) > list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) > list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS winner,
            CASE 
                WHEN list_sum(p1_payoff) < list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) < list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS Looser,
            list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
        FROM sample_second
        UNION ALL
        SELECT 
            p1_name,
            p2_name,
            list_sum(p1_payoff) AS p1_points,
            list_sum(p2_payoff) AS p2_points,
            CASE 
                WHEN list_sum(p1_payoff) > list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) > list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS winner,
            CASE 
                WHEN list_sum(p1_payoff) < list_sum(p2_payoff) THEN p1_name
                WHEN list_sum(p2_payoff) < list_sum(p1_payoff) THEN p2_name
                ELSE NULL
            END AS Looser,
            list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
        FROM sample_third
    )
    WHERE Winner = '{player}'
    GROUP BY Looser, Winner
    ORDER BY perdidos DESC
    """)

┌────────────────────────┬─────────────────┬──────────┐
│         Looser         │     winner      │ perdidos │
│        varchar         │     varchar     │  int64   │
├────────────────────────┼─────────────────┼──────────┤
│ BinarySunset           │ Adaptive Pavlov │       20 │
│ Permissive Tit for Tat │ Adaptive Pavlov │       12 │
│ Uniform Random         │ Adaptive Pavlov │       11 │
│ Contrite TFT           │ Adaptive Pavlov │       10 │
│ Generous TFT           │ Adaptive Pavlov │        6 │
│ Always 0               │ Adaptive Pavlov │        2 │
│ Castigador Infernal    │ Adaptive Pavlov │        1 │
└────────────────────────┴─────────────────┴──────────┘

### Empates

In [58]:
duckdb.query(f"""
        SELECT  p1_name, p2_name, list_sum(p1_payoff) AS p1_points, list_sum(p2_payoff) AS p2_points,  list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
        FROM sample_first
        WHERE is_draw = true
    """)

┌────────────────────┬────────────────────────────┬───────────┬───────────┬─────────┐
│      p1_name       │          p2_name           │ p1_points │ p2_points │ is_draw │
│      varchar       │          varchar           │  int128   │  int128   │ boolean │
├────────────────────┼────────────────────────────┼───────────┼───────────┼─────────┤
│ Adaptive Pavlov    │ Agente Astuto              │         2 │         2 │ true    │
│ Adaptive Pavlov    │ BinarySunset               │         0 │         0 │ true    │
│ Adaptive Pavlov    │ CopyCat                    │        63 │        63 │ true    │
│ Adaptive Pavlov    │ Deterministic Simpletron   │        80 │        80 │ true    │
│ Adaptive Pavlov    │ Deterministic Simpletron   │        36 │        36 │ true    │
│ Adaptive Pavlov    │ Generous TFT               │        28 │        28 │ true    │
│ Adaptive Pavlov    │ Weighted Random 2, 3, or 4 │         8 │         8 │ true    │
│ Agente Astuto      │ Deterministic Simpletron   │   

In [66]:
duckdb.query(f"""
        SELECT player_name , count() as empates
        FROM (
            SELECT  p1_name as player_name, list_sum(p1_payoff) AS p1_points, list_sum(p2_payoff) AS p2_points,  list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
            FROM sample_first
            WHERE is_draw = true
            UNION ALL 
            SELECT  p2_name as player_name, list_sum(p1_payoff) AS p1_points, list_sum(p2_payoff) AS p2_points,  list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
            FROM sample_first
            WHERE is_draw = true
            UNION ALL
            SELECT  p1_name as player_name, list_sum(p1_payoff) AS p1_points, list_sum(p2_payoff) AS p2_points,  list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
            FROM sample_second
            WHERE is_draw = true
            UNION ALL 
            SELECT  p2_name as player_name, list_sum(p1_payoff) AS p1_points, list_sum(p2_payoff) AS p2_points,  list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
            FROM sample_second
            WHERE is_draw = true
            UNION ALL
            SELECT  p1_name as player_name, list_sum(p1_payoff) AS p1_points, list_sum(p2_payoff) AS p2_points,  list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
            FROM sample_third
            WHERE is_draw = true
            UNION ALL 
            SELECT  p2_name as player_name, list_sum(p1_payoff) AS p1_points, list_sum(p2_payoff) AS p2_points,  list_sum(p1_payoff) = list_sum(p2_payoff) AS is_draw
            FROM sample_third
            WHERE is_draw = true
        )
        GROUP BY player_name
        ORDER BY empates DESC
    """)

┌────────────────────────────┬─────────┐
│        player_name         │ empates │
│          varchar           │  int64  │
├────────────────────────────┼─────────┤
│ CopyCat                    │     230 │
│ Generous TFT               │     136 │
│ Contrite TFT               │     126 │
│ Tit for Tat                │     123 │
│ Permissive Tit for Tat     │     121 │
│ BinarySunset               │     105 │
│ Agente Astuto              │      92 │
│ Adaptive Pavlov            │      86 │
│ Deterministic Simpletron   │      82 │
│ Weighted Random 2, 3, or 4 │      60 │
│ WSLS                       │      42 │
│ Random 2 or 3              │      42 │
│ Castigador Infernal        │      34 │
│ Hat Tricker                │      28 │
│ Grim Trigger               │      28 │
│ Detective Avanzado         │      26 │
│ Focal 5                    │      23 │
│ Always 3                   │      19 │
│ Uniform Random             │       5 │
├────────────────────────────┴─────────┤
│ 19 rows       

## Evolution

In [70]:
duckdb.query(f"""
        SELECT *
        FROM sample_second
    """).pl()

game_number,repetition,num_rounds,p1_name,p2_name,p1_action,p2_action,p1_payoff,p2_payoff,mean_score_p1,mean_score_p2,generation
i64,i64,i64,str,str,list[i64],list[i64],list[i64],list[i64],f64,f64,i64
1,1,6,"""Adaptive Pavlov""","""Agente Astuto""","[2, 2, … 2]","[2, 3, … 3]","[2, 2, … 2]","[2, 3, … 3]",2.0,2.5,1
2,2,111,"""Adaptive Pavlov""","""Agente Astuto""","[2, 2, … 2]","[2, 3, … 3]","[2, 2, … 2]","[2, 3, … 3]",1.981982,2.153153,1
3,1,5,"""Adaptive Pavlov""","""BinarySunset""","[2, 3, … 2]","[4, 4, … 4]","[0, 0, … 0]","[0, 0, … 0]",0.0,0.0,1
4,2,23,"""Adaptive Pavlov""","""BinarySunset""","[2, 3, … 3]","[4, 4, … 4]","[0, 0, … 0]","[0, 0, … 0]",0.130435,0.086957,1
5,1,97,"""Adaptive Pavlov""","""Contrite TFT""","[2, 2, … 2]","[2, 2, … 2]","[2, 2, … 2]","[2, 2, … 2]",1.989691,1.989691,1
…,…,…,…,…,…,…,…,…,…,…,…
152,2,4,"""Random 2 or 3""","""Weighted Random 2, 3, or 4""","[2, 2, … 2]","[2, 3, … 3]","[2, 2, … 2]","[2, 3, … 3]",2.25,2.5,10
153,1,83,"""Random 2 or 3""","""WSLS""","[3, 3, … 2]","[2, 3, … 3]","[3, 0, … 2]","[2, 0, … 3]",1.566265,1.783133,10
154,2,146,"""Random 2 or 3""","""WSLS""","[2, 3, … 3]","[2, 3, … 3]","[2, 0, … 0]","[2, 0, … 0]",1.40411,1.60274,10


In [81]:
duckdb.query(f"""
        SELECT DISTINCT(p1_name, p2_name)
        FROM sample_second
        WHERE repetition = 1 and generation = 1
    """).pl()

"main.""row""(p1_name, p2_name)"
struct[2]
"{""Adaptive Pavlov"",""CopyCat""}"
"{""BinarySunset"",""Grim Trigger""}"
"{""BinarySunset"",""Random 2 or 3""}"
"{""Hat Tricker"",""Permissive Tit for Tat""}"
"{""Adaptive Pavlov"",""Agente Astuto""}"
…
"{""Random 2 or 3"",""Weighted Random 2, 3, or 4""}"
"{""Adaptive Pavlov"",""Detective Avanzado""}"
"{""Adaptive Pavlov"",""Weighted Random 2, 3, or 4""}"


# Global

In [10]:
dataframes = []
for file in files:
    match = regex.match(file)
    if match:
        edition = match.group(1)
        phase = match.group(2)
        var_name = f"df_{edition}_{phase}"
        dataframes.append(var_name)
        globals()[var_name] = pl.read_parquet(f'{path}{file}')

# Acceso:
df_20251207_110916_first_phase  # variable creada dinámicamente

game_number,repetition,num_rounds,p1_name,p2_name,p1_action,p2_action,p1_payoff,p2_payoff,mean_score_p1,mean_score_p2
i64,i64,i64,str,str,list[i64],list[i64],list[i64],list[i64],f64,f64
1,1,1,"""Adaptive Pavlov""","""Agente Astuto""",[2],[2],[2],[2],2.0,2.0
2,2,35,"""Adaptive Pavlov""","""Agente Astuto""","[2, 2, … 2]","[2, 2, … 2]","[2, 2, … 2]","[2, 2, … 2]",1.971429,2.142857
3,1,100,"""Adaptive Pavlov""","""BinarySunset""","[2, 3, … 3]","[4, 4, … 2]","[0, 0, … 3]","[0, 0, … 2]",1.62,1.29
4,2,10,"""Adaptive Pavlov""","""BinarySunset""","[2, 3, … 3]","[4, 4, … 4]","[0, 0, … 0]","[0, 0, … 0]",0.0,0.0
5,1,49,"""Adaptive Pavlov""","""Contrite TFT""","[2, 2, … 2]","[2, 2, … 2]","[2, 2, … 2]","[2, 2, … 2]",2.0,2.020408
…,…,…,…,…,…,…,…,…,…,…
178,2,49,"""Random 2 or 3""","""Weighted Random 2, 3, or 4""","[2, 3, … 3]","[3, 2, … 2]","[2, 3, … 3]","[3, 2, … 2]",1.857143,1.755102
179,1,22,"""Random 2 or 3""","""WSLS""","[3, 3, … 2]","[2, 3, … 3]","[3, 0, … 2]","[2, 0, … 3]",1.5,1.5
180,2,48,"""Random 2 or 3""","""WSLS""","[3, 3, … 3]","[2, 3, … 3]","[3, 0, … 0]","[2, 0, … 0]",1.5,1.75


In [13]:
super_query_elements = []
for dataframe_name in dataframes:
    super_query_elements.append(f"""
    SELECT p1_name as player_name, list_sum(p1_payoff) as player_score, num_rounds
    FROM {dataframe_name}
    UNION ALL
    SELECT p2_name as player_name, list_sum(p2_payoff) as player_score, num_rounds
    FROM {dataframe_name}
    """
    )
super_query = "\nUNION ALL\n".join(super_query_elements)

In [15]:
# Total
duckdb.query(f"""
SELECT player_name, SUM(player_score) as total_score, SUM(num_rounds) as total_rounds, total_score/total_rounds as Average
FROM (
    {super_query}
    )
GROUP BY player_name
ORDER BY Average DESC
""")

┌────────────────────────────┬─────────────┬──────────────┬───────────────────────┐
│        player_name         │ total_score │ total_rounds │        Average        │
│          varchar           │   int128    │    int128    │        double         │
├────────────────────────────┼─────────────┼──────────────┼───────────────────────┤
│ Adaptive Pavlov            │      857785 │       438358 │    1.9568138370920571 │
│ Focal 5                    │      470502 │       245677 │     1.915124329912853 │
│ Castigador Infernal        │      453862 │       241113 │     1.882362211908939 │
│ Hat Tricker                │      825744 │       450168 │     1.834301860638695 │
│ Weighted Random 2, 3, or 4 │      805074 │       445026 │    1.8090493589138612 │
│ Contrite TFT               │      723097 │       434906 │    1.6626512395782076 │
│ Random 2 or 3              │      736384 │       443828 │    1.6591652622186974 │
│ WSLS                       │      719322 │       437877 │    1.64274899115